# Deliverable 2 — Full simulator results, noise model, ablations

**Project:** Which data encoding (angle, IQP/ZZ, amplitude) gives the best accuracy per qubit and per gate on standard datasets?

**Author:** Rafay Dil (25280068)

This notebook uses `qenc.py`, `folds/folds.json` and `results/baselines.csv` from Deliverable 1 (`01_pipeline_baselines_first_model.ipynb`) without change. Run that notebook first.

Contents:

1. **Main experiment** — 3 encodings × 3 datasets × 5 fixed folds × 5 seeds on the noiseless simulator.
2. **Resource efficiency** — accuracy per qubit and per two-qubit gate.
3. **Training stability** — spread over seeds, loss curves, initial gradient norms.
4. **Noise model** — depolarising gate noise, read-out error and finite shots applied to the trained models.
5. **Ablations** — ansatz depth, qubit budget, feature-map design and scaling, ansatz entanglement, amplitude preprocessing.


In [ ]:
import json, os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml

import qenc
qenc.set_style()
pd.set_option("display.precision", 3); pd.set_option("display.width", 200)
os.makedirs("figures", exist_ok=True)

FORCE, N_JOBS = False, 2
LAYERS, EPOCHS, LR = 3, 150, 0.05               # defaults fixed in Deliverable 1
MAIN = {"angle": 4, "zz": 4, "amplitude": 2}     # qubits that load d = 4 features
SEEDS = [0, 1, 2, 3, 4]
ABL_SEEDS = [0, 1, 2]
DS, ENC = list(qenc.DATASETS), list(qenc.ENCODINGS)

folds = qenc.load_folds("folds/folds.json")
baselines = pd.read_csv("results/baselines.csv")
base_mean = baselines.groupby(["model", "features", "dataset"])["test_acc"].mean()
majority = base_mean["majority"]["-"]
print("pennylane", qml.__version__)

def cfg(dataset, fold, encoding, n_qubits, seed, **kw):
    return dict(dataset=dataset, fold=fold, encoding=encoding, n_qubits=n_qubits, seed=seed,
                **{"layers": LAYERS, "epochs": EPOCHS, "lr": LR, **kw})

def cv_summary(df, by):
    # Cross-validated accuracy per seed (mean over the 5 folds), then mean and s.d. over seeds.
    per_seed = df.groupby(by + ["seed"])["test_acc"].mean()
    out = per_seed.groupby(by).agg(acc="mean", seed_sd=lambda s: s.std(ddof=1), worst_seed="min", best_seed="max")
    out["fold_sd"] = df.groupby(by + ["fold"])["test_acc"].mean().groupby(by).std(ddof=1)
    out["train_acc"] = df.groupby(by)["train_acc"].mean()
    return out

def pm(mean, sd):
    return f"{mean:.3f} ± {sd:.3f}"

def wide(summary, col_level="dataset", value=None):
    s = summary.apply(lambda r: pm(r["acc"], r["seed_sd"]), axis=1) if value is None else summary[value]
    t = s.unstack(col_level)

## 1. Main experiment


In [ ]:
configs = [cfg(ds, k, enc, nq, s, keep_model=True)
           for ds in DS for enc, nq in MAIN.items() for k in range(qenc.N_FOLDS) for s in SEEDS]
main = qenc.run_grid(configs, folds, "results/main.csv", n_jobs=N_JOBS, force=FORCE)
print(len(main), "runs,", f"{main.train_time_s.sum() / 60:.1f} CPU-minutes of training")
main_sum = cv_summary(main, ["encoding", "dataset"])
main_sum.to_csv("results/main_summary.csv")

In [ ]:
tab = wide(main_sum).loc[ENC]
tab.index = [qenc.LABELS[e] + f" ({MAIN[e]} qubits)" for e in ENC]
for m, f, lab in (("logreg", "PCA-4", "Logistic regression (PCA-4)"), ("svm_rbf", "PCA-4", "RBF SVM (PCA-4)"),
                  ("logreg", "all", "Logistic regression (all features)"), ("svm_rbf", "all", "RBF SVM (all features)"),
                  ("majority", "-", "Majority class")):
    tab.loc[lab] = [f"{base_mean[m][f][d]:.3f}" for d in DS]

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
width = 0.24
for i, enc in enumerate(ENC):
    mu = [main_sum.loc[(enc, d), "acc"] for d in DS]; sd = [main_sum.loc[(enc, d), "seed_sd"] for d in DS]
    xs = np.arange(3) + (i - 1) * (width + 0.02)
    ax.bar(xs, mu, width, yerr=sd, color=qenc.COLORS[enc], label=f"{qenc.LABELS[enc]} ({MAIN[enc]} qubits)",
           error_kw={"lw": 0.8, "capsize": 2, "ecolor": "#0b0b0b"})
    for x_, m_ in zip(xs, mu):
        ax.text(x_, 0.03, f"{m_:.3f}", ha="center", va="bottom", fontsize=7.5, color="white", rotation=90)
for j, d in enumerate(DS):
    ax.hlines(base_mean["logreg"]["PCA-4"][d], j - 0.42, j + 0.42, color="#0b0b0b", lw=1.2, ls="-",
              label="Logistic regression (PCA-4)" if j == 0 else None)
    ax.hlines(majority[d], j - 0.42, j + 0.42, color="#0b0b0b", lw=1.0, ls=":", label="Majority class" if j == 0 else None)
ax.set_xticks(np.arange(3)); ax.set_xticklabels([qenc.LABELS[d] for d in DS]); ax.set_ylim(0, 1.05)
ax.set_ylabel("test accuracy (mean ± s.d. over 5 seeds)")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.28), ncol=3, fontsize=8)

## 2. Accuracy per qubit and per gate

Resource counts are for the full circuit (encoding + ansatz) compiled to $\{R_X,R_Y,R_Z,H,\mathrm{CNOT}\}$. Raw accuracy divided by a resource count rewards small circuits even at chance level, so two views are reported:

* **accuracy / qubit** and **accuracy / two-qubit gate** — the literal ratios;

In [ ]:
def resources(dataset, encoding, n_qubits, layers=LAYERS, **kw):
    Xtr, ytr, Xte, yte = qenc.get_split(dataset, 0, folds)
    P, _ = qenc.prepare(Xtr, Xte, encoding, n_qubits)
    n = 20 if encoding == "amplitude" else 1
    recs = pd.DataFrame([qenc.count_resources(encoding, n_qubits, layers, x, **kw) for x in P[:n]])
    return {"dataset": dataset, **recs.max(numeric_only=True).astype(int).to_dict(), "encoding": encoding}

res = pd.DataFrame([resources(d, e, q) for d in DS for e, q in MAIN.items()]).set_index(["encoding", "dataset"])
eff = main_sum[["acc", "seed_sd"]].join(res[["n_qubits", "total_2q", "enc_2q", "total_depth"]])
eff["majority"] = [majority[d] for d in eff.index.get_level_values("dataset")]
eff["lift"] = eff["acc"] - eff["majority"]
eff["acc_per_qubit"] = eff["acc"] / eff["n_qubits"]
eff["acc_per_2q"] = eff["acc"] / eff["total_2q"]
eff["lift_per_qubit"] = eff["lift"] / eff["n_qubits"]
eff["lift_per_2q"] = eff["lift"] / eff["total_2q"]
eff = eff.loc[[(e, d) for d in DS for e in ENC]]
eff.to_csv("results/efficiency.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.9), sharey=True)
for ax, d in zip(axes, DS):
    for enc in ENC:
        r = eff.loc[(enc, d)]
        ax.errorbar(r["total_2q"], r["acc"], yerr=r["seed_sd"], fmt="o", ms=7, color=qenc.COLORS[enc],
                    ecolor="#0b0b0b", elinewidth=0.8, capsize=2, label=qenc.LABELS[enc])
        ax.annotate(f"{qenc.LABELS[enc]}\n{int(r['n_qubits'])} qubits", (r["total_2q"], r["acc"]),
                    textcoords="offset points", xytext={"angle": (8, 4), "zz": (-36, 9), "amplitude": (-2, -24)}[enc], fontsize=7.5, color="#52514e")
    ax.axhline(majority[d], color="#0b0b0b", lw=1.0, ls=":")
    ax.text(0.5, majority[d] + 0.012, "majority class", fontsize=7, color="#52514e")
    ax.set_title(qenc.LABELS[d]); ax.set_xlabel("two-qubit gates (encoding + ansatz)"); ax.set_xlim(0, 44); ax.set_ylim(0.3, 1.02)
axes[0].set_ylabel("test accuracy"); axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), ncol=3, fontsize=8)

## 3. Training stability


In [ ]:
stab = main_sum[["acc", "seed_sd", "worst_seed", "best_seed"]].copy()
stab["seed_range"] = stab["best_seed"] - stab["worst_seed"]
g = main.groupby(["encoding", "dataset"])
stab["final_loss"] = g["final_loss"].mean()
stab["loss_sd_within_fold"] = main.groupby(["encoding", "dataset", "fold"])["final_loss"].std(ddof=1).groupby(["encoding", "dataset"]).mean()
stab["grad_norm_init"] = g["grad_norm_init"].mean()
stab["train_minus_test"] = g["train_acc"].mean() - g["test_acc"].mean()
stab = stab.loc[[(e, d) for d in DS for e in ENC]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.7))
for ax, d in zip(axes, DS):
    for enc in ENC:
        L = np.array([json.loads(s) for s in main[(main.dataset == d) & (main.encoding == enc)]["losses"]])
        ax.fill_between(range(L.shape[1]), L.min(0), L.max(0), color=qenc.COLORS[enc], alpha=0.18, lw=0)
        ax.plot(L.mean(0), color=qenc.COLORS[enc], label=qenc.LABELS[enc])
    ax.set_title(qenc.LABELS[d]); ax.set_xlabel("epoch")
axes[0].set_ylabel("training cross-entropy"); axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), ncol=3, fontsize=8)
fig.suptitle("Mean training loss; band = min–max over 25 runs (5 folds × 5 seeds)", x=0.72, y=1.03, fontsize=9)
fig.savefig("figures/d2_loss_curves.png"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.5), sharey=True)
per_seed = main.groupby(["dataset", "encoding", "seed"])["test_acc"].mean()
for ax, d in zip(axes, DS):
    for i, enc in enumerate(ENC):
        v = per_seed[d][enc].values
        ax.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, s=28, color=qenc.COLORS[enc],
                   edgecolor="white", linewidth=0.8, zorder=3)
        ax.hlines(v.mean(), i - 0.25, i + 0.25, color="#0b0b0b", lw=1.2)
    ax.set_xticks(range(3)); ax.set_xticklabels([qenc.LABELS[e] for e in ENC]); ax.set_title(qenc.LABELS[d]); ax.set_xlim(-0.5, 2.5)
axes[0].set_ylabel("5-fold test accuracy per seed")
fig.suptitle("One dot per seed; bar = mean", y=1.05, fontsize=9)

## 4. Noise model

The trained models (all 225, weights unchanged) are re-evaluated on their test folds with a density-matrix simulation (`default.mixed`). Circuits are first compiled to $\{R_X,R_Y,R_Z,H,\mathrm{CNOT}\}$; amplitude encoding is compiled with the Möttönen state preparation, so its state-preparation gates are noisy too.

| error source | channel | base rate ($\lambda=1$) |
|---|---|---|
| one-qubit gate | depolarising after every gate | $p_1 = 10^{-3}$ |
| CNOT | depolarising on both qubits after every CNOT | $p_2 = 10^{-2}$ |
| read-out | bit flip before measuring each read-out qubit | $p_m = 2\times10^{-2}$ |
| sampling | 1024 shots per test point | — |


In [ ]:
# consistency check: at lambda = 0 the density-matrix pipeline must reproduce the noiseless accuracy
chk = pd.DataFrame([qenc.noisy_eval(r, folds, 0.0) for _, r in main[(main.fold == 0) & (main.seed == 0)].iterrows()])
assert np.allclose(chk.acc_noisy, chk.acc_ideal), "noise pipeline disagrees with state-vector results"
print("lambda = 0 reproduces the noiseless test accuracy for all", len(chk), "checked models")

LAMS = [0.5, 1, 2, 4, 8]
noise = qenc.noise_grid(main, folds, LAMS, "results/noise.csv", n_jobs=N_JOBS, force=FORCE)
ideal = main.groupby(["encoding", "dataset", "seed"])["test_acc"].mean().groupby(["encoding", "dataset"]).agg(["mean", "std"])
nz = noise.groupby(["encoding", "dataset", "lam", "seed"])[["acc_noisy", "acc_noisy_shots"]].mean()
noise_sum = nz.groupby(["encoding", "dataset", "lam"]).agg(["mean", "std"])
noise_sum.columns = ["exact", "exact_sd", "shots", "shots_sd"]
noise_sum["ideal"] = [ideal.loc[(e, d), "mean"] for e, d, _ in noise_sum.index]
noise_sum["drop_exact"] = noise_sum["ideal"] - noise_sum["exact"]
noise_sum["drop_shots"] = noise_sum["ideal"] - noise_sum["shots"]
noise_sum.to_csv("results/noise_summary.csv")
at1 = noise_sum.xs(1.0, level="lam").loc[[(e, d) for d in DS for e in ENC]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.9), sharey=True)
for ax, d in zip(axes, DS):
    for enc in ENC:
        s = noise_sum.loc[(enc, d)]
        xs = [0] + list(s.index); ys = [ideal.loc[(enc, d), "mean"]] + list(s["shots"]); es = [ideal.loc[(enc, d), "std"]] + list(s["shots_sd"])
        ax.errorbar(range(len(xs)), ys, yerr=es, marker="o", color=qenc.COLORS[enc], ecolor=qenc.COLORS[enc],
                    elinewidth=0.8, capsize=2, label=qenc.LABELS[enc])
    ax.axhline(majority[d], color="#0b0b0b", lw=1.0, ls=":")
    ax.set_xticks(range(len(xs))); ax.set_xticklabels(["0", "0.5", "1", "2", "4", "8"])
    ax.set_title(qenc.LABELS[d]); ax.set_xlabel("noise scale λ (1 = base rates)"); ax.set_ylim(0.3, 1.02)
axes[0].set_ylabel("test accuracy, 1024 shots"); axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), ncol=3, fontsize=8)
fig.suptitle("Dotted line: majority class", x=0.78, y=1.03, fontsize=9)
fig.savefig("figures/d2_noise.png"); plt.show()

drop = noise_sum["drop_shots"].unstack("lam").loc[[(e, d) for d in DS for e in ENC]]
drop.columns = [f"λ={l:g}" for l in drop.columns]

## 5. Ablations


In [ ]:
main3 = main[main.seed.isin(ABL_SEEDS)]
def grid(name, configs):
    df = qenc.run_grid(configs, folds, f"results/ablation_{name}.csv", n_jobs=N_JOBS, force=FORCE)
    print(f"{name}: {len(df)} runs")

### 5.1 Ansatz depth


In [ ]:
depth = grid("depth", [cfg(d, k, e, q, s, layers=L) for L in (1, 2, 4) for d in DS for e, q in MAIN.items()
                       for k in range(qenc.N_FOLDS) for s in ABL_SEEDS])
depth = pd.concat([depth, main3[depth.columns]])
depth_sum = cv_summary(depth, ["encoding", "layers", "dataset"]); depth_sum.to_csv("results/ablation_depth_summary.csv")
fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.7), sharey=True)
for ax, d in zip(axes, DS):
    for enc in ENC:
        s = depth_sum.xs(d, level="dataset").loc[enc]
        ax.errorbar(s.index, s["acc"], yerr=s["seed_sd"], marker="o", color=qenc.COLORS[enc], elinewidth=0.8, capsize=2, label=qenc.LABELS[enc])
    ax.set_xticks([1, 2, 3, 4]); ax.set_title(qenc.LABELS[d]); ax.set_xlabel("ansatz layers L")
axes[0].set_ylabel("test accuracy"); axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), ncol=3, fontsize=8)
fig.savefig("figures/d2_ablation_depth.png"); plt.show()

### 5.2 Qubit budget


In [ ]:
qb = [cfg(d, k, e, n, s) for e in ("angle", "zz") for n in (2, 3) for d in DS for k in range(qenc.N_FOLDS) for s in ABL_SEEDS]
qb += [cfg(d, k, "amplitude", n, s) for n in (3, 4) for d in ("wine", "breast_cancer") for k in range(qenc.N_FOLDS) for s in ABL_SEEDS]
qubits = grid("qubits", qb)
qubits = pd.concat([qubits, main3[qubits.columns]])
qubits_sum = cv_summary(qubits, ["encoding", "n_qubits", "dataset"])
rq = pd.DataFrame([resources(d, e, n) for (e, n, d) in qubits_sum.index]).set_index(["encoding", "n_qubits", "dataset"])
qubits_sum = qubits_sum.join(rq[["n_features", "total_2q", "total_depth"]])
qubits_sum["n_features"] = [min(f, qenc.load_dataset(d)[0].shape[1]) for f, d in zip(qubits_sum["n_features"], qubits_sum.index.get_level_values("dataset"))]
qubits_sum.to_csv("results/ablation_qubits_summary.csv")
fig, axes = plt.subplots(1, 3, figsize=(9.2, 2.7), sharey=True)
for ax, d in zip(axes, DS):
    for enc in ENC:
        s = qubits_sum.xs(d, level="dataset").loc[enc]
        ax.errorbar(s.index, s["acc"], yerr=s["seed_sd"], marker="o", color=qenc.COLORS[enc], elinewidth=0.8, capsize=2, label=qenc.LABELS[enc])
    ax.axhline(majority[d], color="#0b0b0b", lw=1.0, ls=":")
    ax.set_xticks([2, 3, 4]); ax.set_title(qenc.LABELS[d]); ax.set_xlabel("qubits n")
axes[0].set_ylabel("test accuracy"); axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.12), ncol=3, fontsize=8)
fig.savefig("figures/d2_ablation_qubits.png"); plt.show()
t = qubits_sum.assign(acc_pm=qubits_sum.apply(lambda r: pm(r["acc"], r["seed_sd"]), axis=1))

### 5.3 Feature-map design and input scaling

* **IQP/ZZ design** — repetitions (1 or 2) and entanglement pattern (all pairs or nearest-neighbour chain).

In [ ]:
fm = [cfg(d, k, "zz", 4, s, zz_reps=r, zz_entanglement=ent) for r, ent in ((1, "full"), (1, "linear"), (2, "linear"))
      for d in DS for k in range(qenc.N_FOLDS) for s in ABL_SEEDS]
zzd = grid("zz_design", fm)
zzd = pd.concat([zzd, main3[main3.encoding == "zz"][zzd.columns]])
zz_sum = cv_summary(zzd, ["zz_reps", "zz_entanglement", "dataset"])
rz = {(r, e): qenc.count_resources("zz", 4, LAYERS, np.ones(4), zz_reps=r, zz_entanglement=e) for r in (1, 2) for e in ("full", "linear")}
zz_tab = wide(zz_sum)
zz_tab.insert(0, "total 2q gates", [rz[i]["total_2q"] for i in zz_tab.index]); zz_tab.insert(1, "depth", [rz[i]["total_depth"] for i in zz_tab.index])

In [ ]:
sc = [cfg(d, k, e, 4, s, scale=float(v)) for v in (np.pi / 2, 2 * np.pi) for e in ("angle", "zz")
      for d in DS for k in range(qenc.N_FOLDS) for s in ABL_SEEDS]
scale = grid("scale", sc)
scale = pd.concat([scale, main3[main3.encoding != "amplitude"][scale.columns]])
scale["scale_label"] = scale["scale"].map(lambda v: {1: "π/2", 2: "π", 4: "2π"}[int(round(v / (np.pi / 2)))])
scale_sum = cv_summary(scale, ["encoding", "scale_label", "dataset"]); scale_sum.to_csv("results/ablation_scale_summary.csv")

### 5.4 Entanglement in the ansatz


In [ ]:
ne = grid("no_entangle", [cfg(d, k, e, q, s, entangle=False) for d in DS for e, q in MAIN.items()
                          for k in range(qenc.N_FOLDS) for s in ABL_SEEDS])
ent = pd.concat([ne, main3[ne.columns]])
ent_sum = cv_summary(ent, ["encoding", "entangle", "dataset"]); ent_sum.to_csv("results/ablation_entangle_summary.csv")
t = wide(ent_sum).loc[[(e, b) for e in ENC for b in (True, False)]]

### 5.5 Amplitude-encoding preprocessing

Amplitude encoding normalises every sample, and the read-out probabilities are quadratic in the amplitudes, so $x$ and $-x$ give identical predictions. Three ways to prepare the vector, all on 2 qubits:

* **minmax** (main) — features scaled to $[0,1]$: all amplitudes non-negative, so the sign ambiguity cannot occur;
* **standardize** — zero-mean, unit-variance features: signed amplitudes, sign ambiguity present;

In [ ]:
am = grid("amp_mode", [cfg(d, k, "amplitude", 2, s, amp_mode=m) for m in ("standardize", "bias") for d in DS
                       for k in range(qenc.N_FOLDS) for s in ABL_SEEDS])
amp = pd.concat([am, main3[main3.encoding == "amplitude"][am.columns]])
amp_sum = cv_summary(amp, ["amp_mode", "dataset"]); amp_sum.to_csv("results/ablation_amp_mode_summary.csv")
t = wide(amp_sum).loc[["minmax", "standardize", "bias"]]
t2 = wide(amp_sum, value="train_acc").loc[["minmax", "standardize", "bias"]].round(3).add_suffix(" (train)")

In [ ]:
rank = eff.reset_index()
lines = []
for metric, label in (("acc", "accuracy"), ("lift_per_qubit", "lift per qubit"), ("lift_per_2q", "lift per two-qubit gate")):
    best = rank.loc[rank.groupby("dataset")[metric].idxmax()].set_index("dataset")["encoding"]
    worst = rank.loc[rank.groupby("dataset")[metric].idxmin()].set_index("dataset")["encoding"]
    lines.append({"metric": label, **{f"best, {qenc.LABELS[d]}": qenc.LABELS[best[d]] for d in DS},
                  **{f"worst, {qenc.LABELS[d]}": qenc.LABELS[worst[d]] for d in DS}})
overall = eff.groupby("encoding")[["acc", "lift_per_qubit", "lift_per_2q"]].mean().loc[ENC]
overall["drop at λ=1 (1024 shots)"] = noise_sum.xs(1.0, level="lam")["drop_shots"].groupby("encoding").mean()
overall["seed s.d."] = main_sum["seed_sd"].groupby("encoding").mean()
overall.to_csv("results/overall.csv")

The interpretation of these tables is written up in `reports/Report2_Results_Draft.pdf`. The best and the worst encoding identified here are the two candidates for the hardware comparison in the next stage of the project.